# ZGLMM159 — M1 — Análise Exploratória

**Tabela:** `prd_procurement.corp_curated.vw_zglmm159_m1`
**Ambiente:** 🔴 **PRODUÇÃO (PRD)** · **Objeto:** VIEW
**Domínio:** Dados MRP 1 por material, centro e deposito (lote, estoque maximo, ponto de reabastecimento)
**Colunas:** 26 · **Parte M1** de 3 (C1 · M1 · M2)

---

## Como usar

Aperte **Run All**. Todas as células são **independentes** e **somente leitura** — nenhum `CREATE`,
`INSERT` ou gravação. Sem widget, sem view temporária, sem ordem obrigatória.

## Alertas de partida _(dos comentários do `DESCRIBE` e da validação anterior)_

| Severidade | Alerta |
|---|---|
| 🔴 ALTA | **Agora tem `cod_deposito`.** Na validação anterior a M1 colapsava vários depósitos em 1 linha (8.254 linhas perdidas no centro 4058). Verificar na seção 18.1. |
| 🔴 ALTA | **5 colunas declaradas sem fonte** (*"Sem fonte nas cinco cleansed; NULL"*): `cod_custos_armazenagem`, `cod_ciclo_mrp`, `pct_refugo`, `vl_custos_fixos_por_pedido`, `qt_tempo_ciclo_trabalho`. Estarão 100% nulas — no SAP podem ter valor. **Limitação de escopo** a registrar, não erro de carga (seção 6.1). |
| 🟡 MÉDIA | **`ind_marcado_exclusao` preservado sem filtro** — a view traz também itens marcados para eliminação. Se a ZGLMM159 no SAP filtra eliminados, a comparação acusará sobra (seção 16). |
| 🟡 MÉDIA | **Nenhuma coluna de auditoria de carga** (sem `dateingest`/`dh_carga`) — não é possível saber a data do snapshot. Anotar data/hora da consulta. |
| ⚪ BAIXA | **Objeto é VIEW** — `DESCRIBE DETAIL`/`HISTORY` comentados para não quebrar o Run All. |

## Seções

| # | Conteúdo |
|---|---|
| 1 | Metadados |
| 2 | Volumetria |
| 3 | Distribuição por centro — escolha do cenário |
| 4 | Granularidade e chave real |
| 5 | Duplicidade |
| **6** | **Preenchimento de todas as colunas** |
| 7 | Cardinalidade |
| 8 | Domínio das categóricas |
| 10 | Perfil numérico |
| **11** | **Totais para conciliação** |
| 12 | Datas |
| 13 | Códigos e zeros à esquerda |
| **14** | **Checksum de linha** |
| 15 | Amostra |
| 16 | Distribuição interna |
| **18** | **Análises específicas + cruzamento C1 × M1 × M2** |
| 19 | Amostra ampliada |
| 20 | Resumo |


## 1. Metadados

In [0]:
-- 1. ESTRUTURA
DESCRIBE EXTENDED prd_procurement.corp_curated.vw_zglmm159_m1;

In [0]:
-- 1.1 Detalhe e historico nao se aplicam a view.
-- DESCRIBE DETAIL prd_procurement.corp_curated.vw_zglmm159_m1;
-- DESCRIBE HISTORY prd_procurement.corp_curated.vw_zglmm159_m1 LIMIT 20;
SELECT 'Objeto e uma VIEW — os dois comandos acima nao se aplicam' AS observacao;

## 2. Volumetria

In [0]:
-- 2. VOLUMETRIA
SELECT COUNT(*) AS linhas,
       COUNT(DISTINCT cod_material) AS materiais,
       COUNT(DISTINCT cod_centro) AS centros,
       COUNT(DISTINCT cod_empresa) AS empresas,
       COUNT(DISTINCT CONCAT_WS('|', cod_material, cod_centro, cod_deposito)) AS chaves_material_centro_deposito
FROM prd_procurement.corp_curated.vw_zglmm159_m1;

## 3. Distribuição por centro

Base para o cenário de teste (10 mil a 300 mil linhas). A 3.1 mostra os centros já usados
nos demais testes (4128, 4014) e o 4058 da validação anterior.

In [0]:
-- 3. DISTRIBUICAO POR CENTRO
SELECT COALESCE(NULLIF(trim(cod_centro),''),'(vazio)') AS cod_centro, COUNT(*) AS linhas,
       COUNT(DISTINCT cod_material) AS materiais,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_zglmm159_m1), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000 THEN 'CANDIDATO A CENARIO' ELSE '' END AS sugestao
FROM prd_procurement.corp_curated.vw_zglmm159_m1 GROUP BY 1 ORDER BY linhas DESC LIMIT 60;

In [0]:
-- 3.1 CENTROS DOS DEMAIS TESTES
SELECT cod_centro, COUNT(*) AS linhas, COUNT(DISTINCT cod_material) AS materiais,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_zglmm159_m1), 2) AS pct
FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro IN ('4128', '4014', '4058')
GROUP BY cod_centro ORDER BY cod_centro;

## 4. Granularidade e chave real

In [0]:
-- 4. GRANULARIDADE
WITH t AS (SELECT COUNT(*) AS total FROM prd_procurement.corp_curated.vw_zglmm159_m1),
g AS (
  SELECT 'cod_material + cod_centro' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_material`, `cod_centro` FROM prd_procurement.corp_curated.vw_zglmm159_m1)
UNION ALL
  SELECT 'cod_material + cod_centro + cod_deposito' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `cod_deposito` FROM prd_procurement.corp_curated.vw_zglmm159_m1)
UNION ALL
  SELECT 'cod_material + cod_deposito' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_material`, `cod_deposito` FROM prd_procurement.corp_curated.vw_zglmm159_m1)
UNION ALL
  SELECT 'cod_material + cod_centro + cod_deposito + cod_empresa' AS chave, COUNT(*) AS distintos FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `cod_deposito`, `cod_empresa` FROM prd_procurement.corp_curated.vw_zglmm159_m1)
)
SELECT g.chave, t.total AS linhas, g.distintos, ROUND(t.total / g.distintos, 4) AS linhas_por_chave,
       CASE WHEN g.distintos = t.total THEN 'CHAVE UNICA' ELSE 'NAO UNICA' END AS veredito
FROM g CROSS JOIN t ORDER BY linhas_por_chave;

## 5. Duplicidade

Pela chave esperada `cod_material + cod_centro + cod_deposito`.

In [0]:
-- 5. CHAVES DUPLICADAS
SELECT `cod_material`, `cod_centro`, `cod_deposito`, COUNT(*) AS qtd FROM prd_procurement.corp_curated.vw_zglmm159_m1
GROUP BY `cod_material`, `cod_centro`, `cod_deposito` HAVING COUNT(*) > 1 ORDER BY qtd DESC LIMIT 30;

In [0]:
-- 5.1 O QUE DIFERENCIA AS LINHAS DUPLICADAS
WITH dup AS (SELECT `cod_material`, `cod_centro`, `cod_deposito` FROM prd_procurement.corp_curated.vw_zglmm159_m1 GROUP BY `cod_material`, `cod_centro`, `cod_deposito` HAVING COUNT(*) > 1),
d AS (SELECT c.* FROM prd_procurement.corp_curated.vw_zglmm159_m1 c JOIN dup ON c.`cod_material` <=> dup.`cod_material` AND c.`cod_centro` <=> dup.`cod_centro` AND c.`cod_deposito` <=> dup.`cod_deposito`),
agg AS (SELECT `cod_material`, `cod_centro`, `cod_deposito`,
         COUNT(DISTINCT `cod_empresa`) AS `cod_empresa`,
         COUNT(DISTINCT `tp_grupo_mrp`) AS `tp_grupo_mrp`,
         COUNT(DISTINCT `cod_regra_calculo_tamanho_lote`) AS `cod_regra_calculo_tamanho_lote`,
         COUNT(DISTINCT `tp_mrp`) AS `tp_mrp`,
         COUNT(DISTINCT `cod_planejador_mrp`) AS `cod_planejador_mrp`,
         COUNT(DISTINCT `cod_horizonte_planejamento_fixo`) AS `cod_horizonte_planejamento_fixo`,
         COUNT(DISTINCT `cod_custos_armazenagem`) AS `cod_custos_armazenagem`,
         COUNT(DISTINCT `cod_ciclo_mrp`) AS `cod_ciclo_mrp`,
         COUNT(DISTINCT `cod_abc`) AS `cod_abc`,
         COUNT(DISTINCT `tp_grupo_unidade_medida`) AS `tp_grupo_unidade_medida`,
         COUNT(DISTINCT `cod_status_material_centro`) AS `cod_status_material_centro`,
         COUNT(DISTINCT `cod_perfil_arredondamento`) AS `cod_perfil_arredondamento`,
         COUNT(DISTINCT `pct_refugo`) AS `pct_refugo`,
         COUNT(DISTINCT `qt_tamanho_fixo_lote`) AS `qt_tamanho_fixo_lote`,
         COUNT(DISTINCT `qt_tamanho_maximo_lote`) AS `qt_tamanho_maximo_lote`,
         COUNT(DISTINCT `qt_tamanho_minimo_lote`) AS `qt_tamanho_minimo_lote`,
         COUNT(DISTINCT `qt_valor_arredondamento`) AS `qt_valor_arredondamento`,
         COUNT(DISTINCT `vl_custos_fixos_por_pedido`) AS `vl_custos_fixos_por_pedido`,
         COUNT(DISTINCT `qt_estoque_maximo`) AS `qt_estoque_maximo`,
         COUNT(DISTINCT `qt_ponto_reabastecimento`) AS `qt_ponto_reabastecimento`,
         COUNT(DISTINCT `dt_inicio_validade_status`) AS `dt_inicio_validade_status`,
         COUNT(DISTINCT `qt_tempo_ciclo_trabalho`) AS `qt_tempo_ciclo_trabalho`,
         COUNT(DISTINCT `ind_marcado_exclusao`) AS `ind_marcado_exclusao`
  FROM d GROUP BY `cod_material`, `cod_centro`, `cod_deposito`)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1 THEN 'VARIA - faz parte da chave real' ELSE 'constante' END AS veredito
FROM (SELECT stack(23,
    'cod_empresa', MAX(`cod_empresa`),
    'tp_grupo_mrp', MAX(`tp_grupo_mrp`),
    'cod_regra_calculo_tamanho_lote', MAX(`cod_regra_calculo_tamanho_lote`),
    'tp_mrp', MAX(`tp_mrp`),
    'cod_planejador_mrp', MAX(`cod_planejador_mrp`),
    'cod_horizonte_planejamento_fixo', MAX(`cod_horizonte_planejamento_fixo`),
    'cod_custos_armazenagem', MAX(`cod_custos_armazenagem`),
    'cod_ciclo_mrp', MAX(`cod_ciclo_mrp`),
    'cod_abc', MAX(`cod_abc`),
    'tp_grupo_unidade_medida', MAX(`tp_grupo_unidade_medida`),
    'cod_status_material_centro', MAX(`cod_status_material_centro`),
    'cod_perfil_arredondamento', MAX(`cod_perfil_arredondamento`),
    'pct_refugo', MAX(`pct_refugo`),
    'qt_tamanho_fixo_lote', MAX(`qt_tamanho_fixo_lote`),
    'qt_tamanho_maximo_lote', MAX(`qt_tamanho_maximo_lote`),
    'qt_tamanho_minimo_lote', MAX(`qt_tamanho_minimo_lote`),
    'qt_valor_arredondamento', MAX(`qt_valor_arredondamento`),
    'vl_custos_fixos_por_pedido', MAX(`vl_custos_fixos_por_pedido`),
    'qt_estoque_maximo', MAX(`qt_estoque_maximo`),
    'qt_ponto_reabastecimento', MAX(`qt_ponto_reabastecimento`),
    'dt_inicio_validade_status', MAX(`dt_inicio_validade_status`),
    'qt_tempo_ciclo_trabalho', MAX(`qt_tempo_ciclo_trabalho`),
    'ind_marcado_exclusao', MAX(`ind_marcado_exclusao`)
  ) AS (coluna, max_valores_distintos) FROM agg)
ORDER BY max_valores_distintos DESC, coluna;

## 6. Preenchimento de TODAS as colunas

**Seção mais importante.** Coluna 100% nula que **não** está na lista de "sem fonte" é erro de carga.

In [0]:
-- 6. PREENCHIMENTO
WITH t AS (SELECT COUNT(*) AS total FROM prd_procurement.corp_curated.vw_zglmm159_m1),
perf AS (SELECT stack(26,
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'cod_deposito', 'string', COUNT_IF(`cod_deposito` IS NULL), COUNT_IF(`cod_deposito` IS NOT NULL AND lower(trim(`cod_deposito`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_deposito`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'cod_empresa', 'string', COUNT_IF(`cod_empresa` IS NULL), COUNT_IF(`cod_empresa` IS NOT NULL AND lower(trim(`cod_empresa`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_empresa`) RLIKE '^0+([.,]0+)?$'),
    'tp_grupo_mrp', 'string', COUNT_IF(`tp_grupo_mrp` IS NULL), COUNT_IF(`tp_grupo_mrp` IS NOT NULL AND lower(trim(`tp_grupo_mrp`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_grupo_mrp`) RLIKE '^0+([.,]0+)?$'),
    'cod_regra_calculo_tamanho_lote', 'string', COUNT_IF(`cod_regra_calculo_tamanho_lote` IS NULL), COUNT_IF(`cod_regra_calculo_tamanho_lote` IS NOT NULL AND lower(trim(`cod_regra_calculo_tamanho_lote`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_regra_calculo_tamanho_lote`) RLIKE '^0+([.,]0+)?$'),
    'tp_mrp', 'string', COUNT_IF(`tp_mrp` IS NULL), COUNT_IF(`tp_mrp` IS NOT NULL AND lower(trim(`tp_mrp`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_mrp`) RLIKE '^0+([.,]0+)?$'),
    'cod_planejador_mrp', 'string', COUNT_IF(`cod_planejador_mrp` IS NULL), COUNT_IF(`cod_planejador_mrp` IS NOT NULL AND lower(trim(`cod_planejador_mrp`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_planejador_mrp`) RLIKE '^0+([.,]0+)?$'),
    'cod_horizonte_planejamento_fixo', 'string', COUNT_IF(`cod_horizonte_planejamento_fixo` IS NULL), COUNT_IF(`cod_horizonte_planejamento_fixo` IS NOT NULL AND lower(trim(`cod_horizonte_planejamento_fixo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_horizonte_planejamento_fixo`) RLIKE '^0+([.,]0+)?$'),
    'cod_custos_armazenagem', 'string', COUNT_IF(`cod_custos_armazenagem` IS NULL), COUNT_IF(`cod_custos_armazenagem` IS NOT NULL AND lower(trim(`cod_custos_armazenagem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_custos_armazenagem`) RLIKE '^0+([.,]0+)?$'),
    'cod_ciclo_mrp', 'string', COUNT_IF(`cod_ciclo_mrp` IS NULL), COUNT_IF(`cod_ciclo_mrp` IS NOT NULL AND lower(trim(`cod_ciclo_mrp`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_ciclo_mrp`) RLIKE '^0+([.,]0+)?$'),
    'cod_abc', 'string', COUNT_IF(`cod_abc` IS NULL), COUNT_IF(`cod_abc` IS NOT NULL AND lower(trim(`cod_abc`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_abc`) RLIKE '^0+([.,]0+)?$'),
    'tp_grupo_unidade_medida', 'string', COUNT_IF(`tp_grupo_unidade_medida` IS NULL), COUNT_IF(`tp_grupo_unidade_medida` IS NOT NULL AND lower(trim(`tp_grupo_unidade_medida`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_grupo_unidade_medida`) RLIKE '^0+([.,]0+)?$'),
    'cod_status_material_centro', 'string', COUNT_IF(`cod_status_material_centro` IS NULL), COUNT_IF(`cod_status_material_centro` IS NOT NULL AND lower(trim(`cod_status_material_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_status_material_centro`) RLIKE '^0+([.,]0+)?$'),
    'cod_perfil_arredondamento', 'string', COUNT_IF(`cod_perfil_arredondamento` IS NULL), COUNT_IF(`cod_perfil_arredondamento` IS NOT NULL AND lower(trim(`cod_perfil_arredondamento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_perfil_arredondamento`) RLIKE '^0+([.,]0+)?$'),
    'pct_refugo', 'decimal(5,2)', COUNT_IF(`pct_refugo` IS NULL), 0L, COUNT_IF(`pct_refugo` = 0),
    'qt_tamanho_fixo_lote', 'decimal(18,3)', COUNT_IF(`qt_tamanho_fixo_lote` IS NULL), 0L, COUNT_IF(`qt_tamanho_fixo_lote` = 0),
    'qt_tamanho_maximo_lote', 'decimal(18,3)', COUNT_IF(`qt_tamanho_maximo_lote` IS NULL), 0L, COUNT_IF(`qt_tamanho_maximo_lote` = 0),
    'qt_tamanho_minimo_lote', 'decimal(18,3)', COUNT_IF(`qt_tamanho_minimo_lote` IS NULL), 0L, COUNT_IF(`qt_tamanho_minimo_lote` = 0),
    'qt_valor_arredondamento', 'decimal(18,3)', COUNT_IF(`qt_valor_arredondamento` IS NULL), 0L, COUNT_IF(`qt_valor_arredondamento` = 0),
    'vl_custos_fixos_por_pedido', 'decimal(13,2)', COUNT_IF(`vl_custos_fixos_por_pedido` IS NULL), 0L, COUNT_IF(`vl_custos_fixos_por_pedido` = 0),
    'qt_estoque_maximo', 'decimal(18,3)', COUNT_IF(`qt_estoque_maximo` IS NULL), 0L, COUNT_IF(`qt_estoque_maximo` = 0),
    'qt_ponto_reabastecimento', 'decimal(18,3)', COUNT_IF(`qt_ponto_reabastecimento` IS NULL), 0L, COUNT_IF(`qt_ponto_reabastecimento` = 0),
    'dt_inicio_validade_status', 'string', COUNT_IF(`dt_inicio_validade_status` IS NULL), COUNT_IF(`dt_inicio_validade_status` IS NOT NULL AND lower(trim(`dt_inicio_validade_status`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_inicio_validade_status`) RLIKE '^0+([.,]0+)?$'),
    'qt_tempo_ciclo_trabalho', 'decimal(5,0)', COUNT_IF(`qt_tempo_ciclo_trabalho` IS NULL), 0L, COUNT_IF(`qt_tempo_ciclo_trabalho` = 0),
    'ind_marcado_exclusao', 'string', COUNT_IF(`ind_marcado_exclusao` IS NULL), COUNT_IF(`ind_marcado_exclusao` IS NOT NULL AND lower(trim(`ind_marcado_exclusao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_marcado_exclusao`) RLIKE '^0+([.,]0+)?$')
  ) AS (coluna, tipo, nulos, vazios, zeros) FROM prd_procurement.corp_curated.vw_zglmm159_m1)
SELECT p.coluna, p.tipo, p.nulos, p.vazios, p.zeros,
       t.total - p.nulos - p.vazios - p.zeros AS uteis,
       ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros) / t.total, 2) AS pct_util,
       CASE WHEN p.nulos = t.total THEN '1. 100% NULO'
            WHEN t.total - p.nulos - p.vazios - p.zeros <= 0 THEN '2. SEM VALOR UTIL'
            WHEN (t.total - p.nulos - p.vazios - p.zeros) < t.total*0.01 THEN '3. QUASE VAZIO'
            ELSE '9. ok' END AS veredito,
       CASE WHEN p.coluna IN ('cod_custos_armazenagem', 'cod_ciclo_mrp', 'pct_refugo', 'vl_custos_fixos_por_pedido', 'qt_tempo_ciclo_trabalho') THEN 'declarada sem fonte' ELSE '' END AS observacao
FROM perf p CROSS JOIN t ORDER BY veredito, pct_util, coluna;

## 6.1 Colunas declaradas sem fonte

Confirma que as 5 colunas marcadas *"Sem fonte nas cinco cleansed"* estão realmente nulas.
**Na comparação com o SAP, elas devem ser excluídas** e registradas como limitação de escopo.

In [0]:
-- 6.1 COLUNAS SEM FONTE
SELECT 'cod_custos_armazenagem' AS coluna, COUNT_IF(`cod_custos_armazenagem` IS NOT NULL) AS nao_nulos, COUNT(*) AS total,
       CASE WHEN COUNT_IF(`cod_custos_armazenagem` IS NOT NULL) = 0 THEN 'CONFIRMADO - 100% nula' ELSE 'DIVERGE - ha valor' END AS veredito
  FROM prd_procurement.corp_curated.vw_zglmm159_m1
UNION ALL
SELECT 'cod_ciclo_mrp' AS coluna, COUNT_IF(`cod_ciclo_mrp` IS NOT NULL) AS nao_nulos, COUNT(*) AS total,
       CASE WHEN COUNT_IF(`cod_ciclo_mrp` IS NOT NULL) = 0 THEN 'CONFIRMADO - 100% nula' ELSE 'DIVERGE - ha valor' END AS veredito
  FROM prd_procurement.corp_curated.vw_zglmm159_m1
UNION ALL
SELECT 'pct_refugo' AS coluna, COUNT_IF(`pct_refugo` IS NOT NULL) AS nao_nulos, COUNT(*) AS total,
       CASE WHEN COUNT_IF(`pct_refugo` IS NOT NULL) = 0 THEN 'CONFIRMADO - 100% nula' ELSE 'DIVERGE - ha valor' END AS veredito
  FROM prd_procurement.corp_curated.vw_zglmm159_m1
UNION ALL
SELECT 'vl_custos_fixos_por_pedido' AS coluna, COUNT_IF(`vl_custos_fixos_por_pedido` IS NOT NULL) AS nao_nulos, COUNT(*) AS total,
       CASE WHEN COUNT_IF(`vl_custos_fixos_por_pedido` IS NOT NULL) = 0 THEN 'CONFIRMADO - 100% nula' ELSE 'DIVERGE - ha valor' END AS veredito
  FROM prd_procurement.corp_curated.vw_zglmm159_m1
UNION ALL
SELECT 'qt_tempo_ciclo_trabalho' AS coluna, COUNT_IF(`qt_tempo_ciclo_trabalho` IS NOT NULL) AS nao_nulos, COUNT(*) AS total,
       CASE WHEN COUNT_IF(`qt_tempo_ciclo_trabalho` IS NOT NULL) = 0 THEN 'CONFIRMADO - 100% nula' ELSE 'DIVERGE - ha valor' END AS veredito
  FROM prd_procurement.corp_curated.vw_zglmm159_m1;

## 7. Cardinalidade

In [0]:
-- 7. CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM prd_procurement.corp_curated.vw_zglmm159_m1),
card AS (SELECT stack(26,
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'cod_deposito', 'string', approx_count_distinct(`cod_deposito`),
    'cod_centro', 'string', approx_count_distinct(`cod_centro`),
    'cod_empresa', 'string', approx_count_distinct(`cod_empresa`),
    'tp_grupo_mrp', 'string', approx_count_distinct(`tp_grupo_mrp`),
    'cod_regra_calculo_tamanho_lote', 'string', approx_count_distinct(`cod_regra_calculo_tamanho_lote`),
    'tp_mrp', 'string', approx_count_distinct(`tp_mrp`),
    'cod_planejador_mrp', 'string', approx_count_distinct(`cod_planejador_mrp`),
    'cod_horizonte_planejamento_fixo', 'string', approx_count_distinct(`cod_horizonte_planejamento_fixo`),
    'cod_custos_armazenagem', 'string', approx_count_distinct(`cod_custos_armazenagem`),
    'cod_ciclo_mrp', 'string', approx_count_distinct(`cod_ciclo_mrp`),
    'cod_abc', 'string', approx_count_distinct(`cod_abc`),
    'tp_grupo_unidade_medida', 'string', approx_count_distinct(`tp_grupo_unidade_medida`),
    'cod_status_material_centro', 'string', approx_count_distinct(`cod_status_material_centro`),
    'cod_perfil_arredondamento', 'string', approx_count_distinct(`cod_perfil_arredondamento`),
    'pct_refugo', 'decimal(5,2)', approx_count_distinct(`pct_refugo`),
    'qt_tamanho_fixo_lote', 'decimal(18,3)', approx_count_distinct(`qt_tamanho_fixo_lote`),
    'qt_tamanho_maximo_lote', 'decimal(18,3)', approx_count_distinct(`qt_tamanho_maximo_lote`),
    'qt_tamanho_minimo_lote', 'decimal(18,3)', approx_count_distinct(`qt_tamanho_minimo_lote`),
    'qt_valor_arredondamento', 'decimal(18,3)', approx_count_distinct(`qt_valor_arredondamento`),
    'vl_custos_fixos_por_pedido', 'decimal(13,2)', approx_count_distinct(`vl_custos_fixos_por_pedido`),
    'qt_estoque_maximo', 'decimal(18,3)', approx_count_distinct(`qt_estoque_maximo`),
    'qt_ponto_reabastecimento', 'decimal(18,3)', approx_count_distinct(`qt_ponto_reabastecimento`),
    'dt_inicio_validade_status', 'string', approx_count_distinct(`dt_inicio_validade_status`),
    'qt_tempo_ciclo_trabalho', 'decimal(5,0)', approx_count_distinct(`qt_tempo_ciclo_trabalho`),
    'ind_marcado_exclusao', 'string', approx_count_distinct(`ind_marcado_exclusao`)
  ) AS (coluna, tipo, distintos) FROM prd_procurement.corp_curated.vw_zglmm159_m1)
SELECT c.coluna, c.tipo, c.distintos, ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE WHEN c.distintos <= 1 THEN '1. CONSTANTE' WHEN c.distintos <= 3 THEN '2. muito baixa'
            WHEN c.distintos > t.total * 0.95 THEN '3. candidata a identificador' ELSE '9. normal' END AS classificacao
FROM card c CROSS JOIN t ORDER BY c.distintos;

## 8. Domínio das categóricas

Top 8 valores de cada uma.

In [0]:
-- 8. DOMINIO
(SELECT 'cod_empresa' AS coluna, CAST(`cod_empresa` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_m1 GROUP BY `cod_empresa` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_deposito' AS coluna, CAST(`cod_deposito` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_m1 GROUP BY `cod_deposito` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_grupo_mrp' AS coluna, CAST(`tp_grupo_mrp` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_m1 GROUP BY `tp_grupo_mrp` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_regra_calculo_tamanho_lote' AS coluna, CAST(`cod_regra_calculo_tamanho_lote` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_m1 GROUP BY `cod_regra_calculo_tamanho_lote` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_mrp' AS coluna, CAST(`tp_mrp` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_m1 GROUP BY `tp_mrp` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_planejador_mrp' AS coluna, CAST(`cod_planejador_mrp` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_m1 GROUP BY `cod_planejador_mrp` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_abc' AS coluna, CAST(`cod_abc` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_m1 GROUP BY `cod_abc` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_grupo_unidade_medida' AS coluna, CAST(`tp_grupo_unidade_medida` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_m1 GROUP BY `tp_grupo_unidade_medida` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_status_material_centro' AS coluna, CAST(`cod_status_material_centro` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_m1 GROUP BY `cod_status_material_centro` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_perfil_arredondamento' AS coluna, CAST(`cod_perfil_arredondamento` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_m1 GROUP BY `cod_perfil_arredondamento` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_marcado_exclusao' AS coluna, CAST(`ind_marcado_exclusao` AS STRING) AS valor, COUNT(*) AS qtd
   FROM prd_procurement.corp_curated.vw_zglmm159_m1 GROUP BY `ind_marcado_exclusao` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 10. Perfil numérico

Todos `decimal` — conciliação exata, sem tolerância.

In [0]:
-- 10. PERFIL NUMERICO
SELECT * FROM (SELECT stack(9,
    'pct_refugo', 'decimal(5,2)', COUNT(`pct_refugo`), CAST(MIN(`pct_refugo`) AS DOUBLE), CAST(MAX(`pct_refugo`) AS DOUBLE), CAST(AVG(`pct_refugo`) AS DOUBLE), CAST(percentile_approx(`pct_refugo`, 0.5) AS DOUBLE), COUNT_IF(`pct_refugo` < 0), COUNT_IF(`pct_refugo` = 0),
    'qt_tamanho_fixo_lote', 'decimal(18,3)', COUNT(`qt_tamanho_fixo_lote`), CAST(MIN(`qt_tamanho_fixo_lote`) AS DOUBLE), CAST(MAX(`qt_tamanho_fixo_lote`) AS DOUBLE), CAST(AVG(`qt_tamanho_fixo_lote`) AS DOUBLE), CAST(percentile_approx(`qt_tamanho_fixo_lote`, 0.5) AS DOUBLE), COUNT_IF(`qt_tamanho_fixo_lote` < 0), COUNT_IF(`qt_tamanho_fixo_lote` = 0),
    'qt_tamanho_maximo_lote', 'decimal(18,3)', COUNT(`qt_tamanho_maximo_lote`), CAST(MIN(`qt_tamanho_maximo_lote`) AS DOUBLE), CAST(MAX(`qt_tamanho_maximo_lote`) AS DOUBLE), CAST(AVG(`qt_tamanho_maximo_lote`) AS DOUBLE), CAST(percentile_approx(`qt_tamanho_maximo_lote`, 0.5) AS DOUBLE), COUNT_IF(`qt_tamanho_maximo_lote` < 0), COUNT_IF(`qt_tamanho_maximo_lote` = 0),
    'qt_tamanho_minimo_lote', 'decimal(18,3)', COUNT(`qt_tamanho_minimo_lote`), CAST(MIN(`qt_tamanho_minimo_lote`) AS DOUBLE), CAST(MAX(`qt_tamanho_minimo_lote`) AS DOUBLE), CAST(AVG(`qt_tamanho_minimo_lote`) AS DOUBLE), CAST(percentile_approx(`qt_tamanho_minimo_lote`, 0.5) AS DOUBLE), COUNT_IF(`qt_tamanho_minimo_lote` < 0), COUNT_IF(`qt_tamanho_minimo_lote` = 0),
    'qt_valor_arredondamento', 'decimal(18,3)', COUNT(`qt_valor_arredondamento`), CAST(MIN(`qt_valor_arredondamento`) AS DOUBLE), CAST(MAX(`qt_valor_arredondamento`) AS DOUBLE), CAST(AVG(`qt_valor_arredondamento`) AS DOUBLE), CAST(percentile_approx(`qt_valor_arredondamento`, 0.5) AS DOUBLE), COUNT_IF(`qt_valor_arredondamento` < 0), COUNT_IF(`qt_valor_arredondamento` = 0),
    'vl_custos_fixos_por_pedido', 'decimal(13,2)', COUNT(`vl_custos_fixos_por_pedido`), CAST(MIN(`vl_custos_fixos_por_pedido`) AS DOUBLE), CAST(MAX(`vl_custos_fixos_por_pedido`) AS DOUBLE), CAST(AVG(`vl_custos_fixos_por_pedido`) AS DOUBLE), CAST(percentile_approx(`vl_custos_fixos_por_pedido`, 0.5) AS DOUBLE), COUNT_IF(`vl_custos_fixos_por_pedido` < 0), COUNT_IF(`vl_custos_fixos_por_pedido` = 0),
    'qt_estoque_maximo', 'decimal(18,3)', COUNT(`qt_estoque_maximo`), CAST(MIN(`qt_estoque_maximo`) AS DOUBLE), CAST(MAX(`qt_estoque_maximo`) AS DOUBLE), CAST(AVG(`qt_estoque_maximo`) AS DOUBLE), CAST(percentile_approx(`qt_estoque_maximo`, 0.5) AS DOUBLE), COUNT_IF(`qt_estoque_maximo` < 0), COUNT_IF(`qt_estoque_maximo` = 0),
    'qt_ponto_reabastecimento', 'decimal(18,3)', COUNT(`qt_ponto_reabastecimento`), CAST(MIN(`qt_ponto_reabastecimento`) AS DOUBLE), CAST(MAX(`qt_ponto_reabastecimento`) AS DOUBLE), CAST(AVG(`qt_ponto_reabastecimento`) AS DOUBLE), CAST(percentile_approx(`qt_ponto_reabastecimento`, 0.5) AS DOUBLE), COUNT_IF(`qt_ponto_reabastecimento` < 0), COUNT_IF(`qt_ponto_reabastecimento` = 0),
    'qt_tempo_ciclo_trabalho', 'decimal(5,0)', COUNT(`qt_tempo_ciclo_trabalho`), CAST(MIN(`qt_tempo_ciclo_trabalho`) AS DOUBLE), CAST(MAX(`qt_tempo_ciclo_trabalho`) AS DOUBLE), CAST(AVG(`qt_tempo_ciclo_trabalho`) AS DOUBLE), CAST(percentile_approx(`qt_tempo_ciclo_trabalho`, 0.5) AS DOUBLE), COUNT_IF(`qt_tempo_ciclo_trabalho` < 0), COUNT_IF(`qt_tempo_ciclo_trabalho` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, negativos, zeros) FROM prd_procurement.corp_curated.vw_zglmm159_m1)
ORDER BY coluna;

## 11. Totais para conciliação com o SAP

In [0]:
-- 11. TOTAIS
SELECT * FROM (SELECT stack(9,
    'pct_refugo', CAST(ROUND(SUM(`pct_refugo`), 3) AS STRING), COUNT(`pct_refugo`),
    'qt_tamanho_fixo_lote', CAST(ROUND(SUM(`qt_tamanho_fixo_lote`), 3) AS STRING), COUNT(`qt_tamanho_fixo_lote`),
    'qt_tamanho_maximo_lote', CAST(ROUND(SUM(`qt_tamanho_maximo_lote`), 3) AS STRING), COUNT(`qt_tamanho_maximo_lote`),
    'qt_tamanho_minimo_lote', CAST(ROUND(SUM(`qt_tamanho_minimo_lote`), 3) AS STRING), COUNT(`qt_tamanho_minimo_lote`),
    'qt_valor_arredondamento', CAST(ROUND(SUM(`qt_valor_arredondamento`), 3) AS STRING), COUNT(`qt_valor_arredondamento`),
    'vl_custos_fixos_por_pedido', CAST(ROUND(SUM(`vl_custos_fixos_por_pedido`), 3) AS STRING), COUNT(`vl_custos_fixos_por_pedido`),
    'qt_estoque_maximo', CAST(ROUND(SUM(`qt_estoque_maximo`), 3) AS STRING), COUNT(`qt_estoque_maximo`),
    'qt_ponto_reabastecimento', CAST(ROUND(SUM(`qt_ponto_reabastecimento`), 3) AS STRING), COUNT(`qt_ponto_reabastecimento`),
    'qt_tempo_ciclo_trabalho', CAST(ROUND(SUM(`qt_tempo_ciclo_trabalho`), 3) AS STRING), COUNT(`qt_tempo_ciclo_trabalho`)
  ) AS (coluna, total, linhas_preenchidas) FROM prd_procurement.corp_curated.vw_zglmm159_m1)
ORDER BY coluna;

## 12. Datas

In [0]:
-- 12. DATAS EM STRING
SELECT * FROM (SELECT stack(1,
    'dt_inicio_validade_status', COUNT_IF(`dt_inicio_validade_status` IS NULL OR trim(`dt_inicio_validade_status`) = ''), COUNT_IF(trim(`dt_inicio_validade_status`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_inicio_validade_status`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_inicio_validade_status`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_inicio_validade_status`) = '00000000'), MIN(CASE WHEN trim(`dt_inicio_validade_status`) NOT IN ('', '00000000') THEN `dt_inicio_validade_status` END), MAX(CASE WHEN trim(`dt_inicio_validade_status`) NOT IN ('', '00000000') THEN `dt_inicio_validade_status` END)
  ) AS (coluna, vazios, fmt_AAAAMMDD, fmt_ISO, fmt_BR, data_zero, minimo, maximo) FROM prd_procurement.corp_curated.vw_zglmm159_m1);

## 13. Códigos e zeros à esquerda

Na validação anterior, `cod_material` vinha com zeros à esquerda e o extrato SAP sem — normalizar os dois lados.

In [0]:
-- 13. CODIGOS
SELECT coluna, vazios, len_min, len_max, com_zeros_esq, distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes
FROM (SELECT stack(5,
    'cod_material', COUNT_IF(`cod_material` IS NULL OR trim(`cod_material`) = ''), MIN(length(trim(`cod_material`))), MAX(length(trim(`cod_material`))), COUNT_IF(trim(`cod_material`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`cod_material`)), COUNT(DISTINCT regexp_replace(trim(`cod_material`), '^0+', '')),
    'cod_centro', COUNT_IF(`cod_centro` IS NULL OR trim(`cod_centro`) = ''), MIN(length(trim(`cod_centro`))), MAX(length(trim(`cod_centro`))), COUNT_IF(trim(`cod_centro`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`cod_centro`)), COUNT(DISTINCT regexp_replace(trim(`cod_centro`), '^0+', '')),
    'cod_deposito', COUNT_IF(`cod_deposito` IS NULL OR trim(`cod_deposito`) = ''), MIN(length(trim(`cod_deposito`))), MAX(length(trim(`cod_deposito`))), COUNT_IF(trim(`cod_deposito`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`cod_deposito`)), COUNT(DISTINCT regexp_replace(trim(`cod_deposito`), '^0+', '')),
    'cod_empresa', COUNT_IF(`cod_empresa` IS NULL OR trim(`cod_empresa`) = ''), MIN(length(trim(`cod_empresa`))), MAX(length(trim(`cod_empresa`))), COUNT_IF(trim(`cod_empresa`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`cod_empresa`)), COUNT(DISTINCT regexp_replace(trim(`cod_empresa`), '^0+', '')),
    'cod_planejador_mrp', COUNT_IF(`cod_planejador_mrp` IS NULL OR trim(`cod_planejador_mrp`) = ''), MIN(length(trim(`cod_planejador_mrp`))), MAX(length(trim(`cod_planejador_mrp`))), COUNT_IF(trim(`cod_planejador_mrp`) RLIKE '^0[0-9]'), COUNT(DISTINCT trim(`cod_planejador_mrp`)), COUNT(DISTINCT regexp_replace(trim(`cod_planejador_mrp`), '^0+', ''))
  ) AS (coluna, vazios, len_min, len_max, com_zeros_esq, distintos_bruto, distintos_sem_zeros) FROM prd_procurement.corp_curated.vw_zglmm159_m1)
ORDER BY coluna;

## 14. Checksum de linha

Se `linhas` > `linhas_unicas`, há registros 100% idênticos.

In [0]:
-- 14. CHECKSUM
SELECT COUNT(*) AS linhas,
       COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`tp_grupo_mrp` AS STRING), ''), COALESCE(CAST(`cod_regra_calculo_tamanho_lote` AS STRING), ''), COALESCE(CAST(`tp_mrp` AS STRING), ''), COALESCE(CAST(`cod_planejador_mrp` AS STRING), ''), COALESCE(CAST(`cod_horizonte_planejamento_fixo` AS STRING), ''), COALESCE(CAST(`cod_custos_armazenagem` AS STRING), ''), COALESCE(CAST(`cod_ciclo_mrp` AS STRING), ''), COALESCE(CAST(`cod_abc` AS STRING), ''), COALESCE(CAST(`tp_grupo_unidade_medida` AS STRING), ''), COALESCE(CAST(`cod_status_material_centro` AS STRING), ''), COALESCE(CAST(`cod_perfil_arredondamento` AS STRING), ''), COALESCE(CAST(`pct_refugo` AS STRING), ''), COALESCE(CAST(`qt_tamanho_fixo_lote` AS STRING), ''), COALESCE(CAST(`qt_tamanho_maximo_lote` AS STRING), ''), COALESCE(CAST(`qt_tamanho_minimo_lote` AS STRING), ''), COALESCE(CAST(`qt_valor_arredondamento` AS STRING), ''), COALESCE(CAST(`vl_custos_fixos_por_pedido` AS STRING), ''), COALESCE(CAST(`qt_estoque_maximo` AS STRING), ''), COALESCE(CAST(`qt_ponto_reabastecimento` AS STRING), ''), COALESCE(CAST(`dt_inicio_validade_status` AS STRING), ''), COALESCE(CAST(`qt_tempo_ciclo_trabalho` AS STRING), ''), COALESCE(CAST(`ind_marcado_exclusao` AS STRING), '')))) AS linhas_unicas,
       COUNT(*) - COUNT(DISTINCT md5(CONCAT_WS('|', COALESCE(CAST(`cod_material` AS STRING), ''), COALESCE(CAST(`cod_deposito` AS STRING), ''), COALESCE(CAST(`cod_centro` AS STRING), ''), COALESCE(CAST(`cod_empresa` AS STRING), ''), COALESCE(CAST(`tp_grupo_mrp` AS STRING), ''), COALESCE(CAST(`cod_regra_calculo_tamanho_lote` AS STRING), ''), COALESCE(CAST(`tp_mrp` AS STRING), ''), COALESCE(CAST(`cod_planejador_mrp` AS STRING), ''), COALESCE(CAST(`cod_horizonte_planejamento_fixo` AS STRING), ''), COALESCE(CAST(`cod_custos_armazenagem` AS STRING), ''), COALESCE(CAST(`cod_ciclo_mrp` AS STRING), ''), COALESCE(CAST(`cod_abc` AS STRING), ''), COALESCE(CAST(`tp_grupo_unidade_medida` AS STRING), ''), COALESCE(CAST(`cod_status_material_centro` AS STRING), ''), COALESCE(CAST(`cod_perfil_arredondamento` AS STRING), ''), COALESCE(CAST(`pct_refugo` AS STRING), ''), COALESCE(CAST(`qt_tamanho_fixo_lote` AS STRING), ''), COALESCE(CAST(`qt_tamanho_maximo_lote` AS STRING), ''), COALESCE(CAST(`qt_tamanho_minimo_lote` AS STRING), ''), COALESCE(CAST(`qt_valor_arredondamento` AS STRING), ''), COALESCE(CAST(`vl_custos_fixos_por_pedido` AS STRING), ''), COALESCE(CAST(`qt_estoque_maximo` AS STRING), ''), COALESCE(CAST(`qt_ponto_reabastecimento` AS STRING), ''), COALESCE(CAST(`dt_inicio_validade_status` AS STRING), ''), COALESCE(CAST(`qt_tempo_ciclo_trabalho` AS STRING), ''), COALESCE(CAST(`ind_marcado_exclusao` AS STRING), '')))) AS linhas_100pct_identicas
FROM prd_procurement.corp_curated.vw_zglmm159_m1;

## 15. Amostra

In [0]:
-- 15. AMOSTRA
SELECT * FROM prd_procurement.corp_curated.vw_zglmm159_m1
ORDER BY `cod_material`, `cod_centro`, `cod_deposito`
LIMIT 20;

## 16. Distribuição interna

Inclui `ind_marcado_exclusao` — mostra quanto da view são itens marcados para eliminação.

In [0]:
-- 16. DISTRIBUICAO POR cod_empresa
SELECT COALESCE(NULLIF(trim(`cod_empresa`), ''), '(vazio)') AS `cod_empresa`, COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_zglmm159_m1), 2) AS pct
FROM prd_procurement.corp_curated.vw_zglmm159_m1 GROUP BY 1 ORDER BY linhas DESC LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR tp_mrp
SELECT COALESCE(NULLIF(trim(`tp_mrp`), ''), '(vazio)') AS `tp_mrp`, COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_zglmm159_m1), 2) AS pct
FROM prd_procurement.corp_curated.vw_zglmm159_m1 GROUP BY 1 ORDER BY linhas DESC LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR cod_abc
SELECT COALESCE(NULLIF(trim(`cod_abc`), ''), '(vazio)') AS `cod_abc`, COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_zglmm159_m1), 2) AS pct
FROM prd_procurement.corp_curated.vw_zglmm159_m1 GROUP BY 1 ORDER BY linhas DESC LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR cod_regra_calculo_tamanho_lote
SELECT COALESCE(NULLIF(trim(`cod_regra_calculo_tamanho_lote`), ''), '(vazio)') AS `cod_regra_calculo_tamanho_lote`, COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_zglmm159_m1), 2) AS pct
FROM prd_procurement.corp_curated.vw_zglmm159_m1 GROUP BY 1 ORDER BY linhas DESC LIMIT 40;

In [0]:
-- 16. DISTRIBUICAO POR ind_marcado_exclusao
SELECT COALESCE(NULLIF(trim(`ind_marcado_exclusao`), ''), '(vazio)') AS `ind_marcado_exclusao`, COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM prd_procurement.corp_curated.vw_zglmm159_m1), 2) AS pct
FROM prd_procurement.corp_curated.vw_zglmm159_m1 GROUP BY 1 ORDER BY linhas DESC LIMIT 40;

## 18. Análises específicas — M1 (MRP 1)

### 18.1 Depósitos por material + centro

Na validação anterior, materiais com vários depósitos no SAP eram **colapsados em 1 linha**
(4.779 materiais, 8.254 linhas perdidas no centro 4058). A M1 agora tem `cod_deposito` —
esta distribuição mostra se o nível de detalhe por depósito foi preservado.

In [0]:
-- 18.1 DEPOSITOS POR MATERIAL + CENTRO
SELECT qt_depositos, COUNT(*) AS materiais_centro, SUM(linhas) AS linhas
FROM (SELECT cod_material, cod_centro, COUNT(DISTINCT cod_deposito) AS qt_depositos, COUNT(*) AS linhas
        FROM prd_procurement.corp_curated.vw_zglmm159_m1 GROUP BY cod_material, cod_centro)
GROUP BY qt_depositos ORDER BY qt_depositos;

In [0]:
-- 18.1b CENTRO 4058 (onde o colapso foi encontrado)
SELECT COUNT(*) AS linhas,
       COUNT(DISTINCT cod_material) AS materiais,
       COUNT(DISTINCT CONCAT_WS('|', cod_material, cod_deposito)) AS material_deposito,
       COUNT(DISTINCT CONCAT_WS('|', cod_material, cod_deposito)) - COUNT(DISTINCT cod_material) AS linhas_alem_de_1_por_material
FROM prd_procurement.corp_curated.vw_zglmm159_m1 WHERE cod_centro = '4058';

### 18.2 Coerência dos parâmetros de lote e estoque

In [0]:
-- 18.2 LOTE E ESTOQUE
SELECT 'lote minimo > lote maximo' AS regra,
       COUNT_IF(qt_tamanho_minimo_lote > qt_tamanho_maximo_lote AND qt_tamanho_maximo_lote > 0) AS linhas FROM prd_procurement.corp_curated.vw_zglmm159_m1
UNION ALL SELECT 'ponto de reabastecimento > estoque maximo',
       COUNT_IF(qt_ponto_reabastecimento > qt_estoque_maximo AND qt_estoque_maximo > 0) FROM prd_procurement.corp_curated.vw_zglmm159_m1
UNION ALL SELECT 'regra de lote FX sem tamanho fixo',
       COUNT_IF(trim(cod_regra_calculo_tamanho_lote) = 'FX' AND COALESCE(qt_tamanho_fixo_lote,0) = 0) FROM prd_procurement.corp_curated.vw_zglmm159_m1
UNION ALL SELECT 'regra de lote HB sem estoque maximo',
       COUNT_IF(trim(cod_regra_calculo_tamanho_lote) = 'HB' AND COALESCE(qt_estoque_maximo,0) = 0) FROM prd_procurement.corp_curated.vw_zglmm159_m1
UNION ALL SELECT 'MRP tipo VB sem ponto de reabastecimento',
       COUNT_IF(trim(tp_mrp) = 'VB' AND COALESCE(qt_ponto_reabastecimento,0) = 0) FROM prd_procurement.corp_curated.vw_zglmm159_m1
UNION ALL SELECT 'quantidades negativas',
       COUNT_IF(qt_estoque_maximo < 0 OR qt_ponto_reabastecimento < 0 OR qt_tamanho_fixo_lote < 0) FROM prd_procurement.corp_curated.vw_zglmm159_m1
ORDER BY linhas DESC;

In [0]:
-- 18.3 STATUS DO MATERIAL x DATA DE VALIDADE
SELECT COALESCE(NULLIF(trim(cod_status_material_centro),''),'(vazio)') AS status,
       COUNT(*) AS linhas,
       COUNT_IF(trim(COALESCE(dt_inicio_validade_status,'')) IN ('', '00000000')) AS sem_data_validade,
       MIN(CASE WHEN trim(dt_inicio_validade_status) NOT IN ('', '00000000') THEN dt_inicio_validade_status END) AS data_min,
       MAX(CASE WHEN trim(dt_inicio_validade_status) NOT IN ('', '00000000') THEN dt_inicio_validade_status END) AS data_max
FROM prd_procurement.corp_curated.vw_zglmm159_m1 GROUP BY 1 ORDER BY linhas DESC LIMIT 20;

### 18.9 🔗 Cruzamento entre as 3 views

A ZGLMM159 foi dividida em C1, M1 e M2. As três deveriam cobrir o **mesmo universo de
`material + centro`** e concordar na empresa. Esta célula mede a cobertura cruzada — divergência
aqui indica material que vai "sumir" quando as três forem reunidas no relatório final.

> Na validação anterior, **434 materiais existiam no SAP e não no Datalake**. Esta análise
> ajuda a localizar em qual das três views eles se perdem.

In [0]:
-- 18.9 COBERTURA DE MATERIAL + CENTRO ENTRE C1, M1, M2
WITH c1 AS (SELECT DISTINCT cod_material, cod_centro FROM prd_procurement.corp_curated.vw_zglmm159_c1),
     m1 AS (SELECT DISTINCT cod_material, cod_centro FROM prd_procurement.corp_curated.vw_zglmm159_m1),
     m2 AS (SELECT DISTINCT cod_material, cod_centro FROM prd_procurement.corp_curated.vw_zglmm159_m2),
     u  AS (SELECT * FROM c1 UNION SELECT * FROM m1 UNION SELECT * FROM m2),
     x  AS (SELECT u.cod_material, u.cod_centro,
                   c1.cod_material IS NOT NULL AS em_c1,
                   m1.cod_material IS NOT NULL AS em_m1,
                   m2.cod_material IS NOT NULL AS em_m2
            FROM u
            LEFT JOIN c1 ON u.cod_material = c1.cod_material AND u.cod_centro = c1.cod_centro
            LEFT JOIN m1 ON u.cod_material = m1.cod_material AND u.cod_centro = m1.cod_centro
            LEFT JOIN m2 ON u.cod_material = m2.cod_material AND u.cod_centro = m2.cod_centro)
SELECT CONCAT(CASE WHEN em_c1 THEN 'C1 ' ELSE '-- ' END,
              CASE WHEN em_m1 THEN 'M1 ' ELSE '-- ' END,
              CASE WHEN em_m2 THEN 'M2'  ELSE '--' END) AS presenca,
       COUNT(*) AS material_centro,
       ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
FROM x GROUP BY 1 ORDER BY material_centro DESC;

In [0]:
-- 18.9b EMPRESA E INDICADOR DE EXCLUSAO CONSISTENTES ENTRE AS VIEWS
WITH a AS (SELECT DISTINCT cod_material, cod_centro, cod_empresa, ind_marcado_exclusao FROM prd_procurement.corp_curated.vw_zglmm159_m1),
     b AS (SELECT DISTINCT cod_material, cod_centro, cod_empresa, ind_marcado_exclusao FROM prd_procurement.corp_curated.vw_zglmm159_c1),
     c AS (SELECT DISTINCT cod_material, cod_centro, cod_empresa, ind_marcado_exclusao FROM prd_procurement.corp_curated.vw_zglmm159_m2)
SELECT 'M1 x C1' AS par,
       COUNT(*) AS material_centro_em_comum,
       COUNT_IF(COALESCE(a.cod_empresa,'') <> COALESCE(b.cod_empresa,'')) AS empresa_diverge,
       COUNT_IF(COALESCE(a.ind_marcado_exclusao,'') <> COALESCE(b.ind_marcado_exclusao,'')) AS exclusao_diverge
  FROM a JOIN b ON a.cod_material = b.cod_material AND a.cod_centro = b.cod_centro
UNION ALL
SELECT 'M1 x M2',
       COUNT(*),
       COUNT_IF(COALESCE(a.cod_empresa,'') <> COALESCE(c.cod_empresa,'')),
       COUNT_IF(COALESCE(a.ind_marcado_exclusao,'') <> COALESCE(c.ind_marcado_exclusao,''))
  FROM a JOIN c ON a.cod_material = c.cod_material AND a.cod_centro = c.cod_centro;

## 19. Amostra ampliada

A extração completa fica nos notebooks de cenário.

In [0]:
-- 19. AMOSTRA AMPLIADA
SELECT * FROM prd_procurement.corp_curated.vw_zglmm159_m1
ORDER BY `cod_material`, `cod_centro`, `cod_deposito`
LIMIT 500;

## 20. Resumo

**Copie esta saída** e envie ao agente junto com o notebook.

In [0]:
-- 20. RESUMO
WITH t AS (SELECT COUNT(*) AS total FROM prd_procurement.corp_curated.vw_zglmm159_m1),
g AS (
  SELECT 'cod_material + cod_centro' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_material`, `cod_centro` FROM prd_procurement.corp_curated.vw_zglmm159_m1)
UNION ALL
  SELECT 'cod_material + cod_centro + cod_deposito' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `cod_deposito` FROM prd_procurement.corp_curated.vw_zglmm159_m1)
UNION ALL
  SELECT 'cod_material + cod_deposito' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_material`, `cod_deposito` FROM prd_procurement.corp_curated.vw_zglmm159_m1)
UNION ALL
  SELECT 'cod_material + cod_centro + cod_deposito + cod_empresa' AS chave, COUNT(*) AS d FROM (SELECT DISTINCT `cod_material`, `cod_centro`, `cod_deposito`, `cod_empresa` FROM prd_procurement.corp_curated.vw_zglmm159_m1)
)
SELECT 'IDENTIFICACAO' AS bloco, 'transacao' AS item, 'ZGLMM159 - M1' AS valor
UNION ALL SELECT 'IDENTIFICACAO', 'ambiente', 'PRD'
UNION ALL SELECT 'IDENTIFICACAO', 'tabela', 'vw_zglmm159_m1'
UNION ALL SELECT 'IDENTIFICACAO', 'linhas', format_number((SELECT total FROM t), 0)
UNION ALL SELECT 'IDENTIFICACAO', 'colunas', '26 (5 declaradas sem fonte)'
UNION ALL SELECT 'GRANULARIDADE', g.chave,
       CONCAT(format_number(g.d, 0), ' distintos | ', CAST(ROUND(t.total / g.d, 4) AS STRING), ' linhas/chave | ',
              CASE WHEN g.d = t.total THEN 'CHAVE UNICA' ELSE 'nao unica' END)
  FROM g CROSS JOIN t
UNION ALL SELECT 'CHAVE REAL', 'sugerida',
       COALESCE((SELECT MIN(g.chave) FROM g CROSS JOIN t WHERE g.d = t.total), 'NENHUMA - investigar')
ORDER BY bloco, item;

---

## Próximo passo

1. Rodar os **3 notebooks** (C1, M1, M2) e comparar a seção 18.9 entre eles.
2. Escolher o recorte de teste pela **seção 3/3.1**.
3. Gerar os notebooks de cenário e extrair a ZGLMM159 no **SAP PRD** com o mesmo recorte e na mesma data.

### Checklist antes de comparar

- [ ] Chave real confirmada (seção 4)
- [ ] Colunas sem fonte excluídas da comparação (seção 6.1)
- [ ] Zeros à esquerda normalizados (seção 13)
- [ ] Itens marcados para exclusão tratados conforme o filtro do SAP (seção 16)
- [ ] Cobertura C1 × M1 × M2 conferida (seção 18.9)
